# INDUSMIND AI - Machine Learning Telemetry Classifier & Knowledge Graph Visualizer
### AI-Powered Industrial Knowledge Intelligence Platform

This notebook implements the end-to-end model training, validation, and visual graph linkages mapping:
1. Preprocess raw sensor telemetry (Temperature and Vibration values).
2. Fit a Nearest-Centroid Classifier to recognize status classes: `Operational`, `Maintenance`, and `Degraded`.
3. Save the trained model parameters to standard config JSON.
4. Construct a network model linking machinery assets, technical documents, and warning states.
5. Render the system topology visually using NetworkX and Matplotlib.

In [ ]:
import os
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
# Raw telemetry simulated logs for training
telemetry_data = [
    {"asset": "PUMP-P102", "temp": 42.0, "vibration": 1.2, "status": "Operational"},
    {"asset": "PUMP-P102", "temp": 44.5, "vibration": 1.4, "status": "Operational"},
    {"asset": "BOILER-B401", "temp": 210.0, "vibration": 0.8, "status": "Operational"},
    {"asset": "BOILER-B401", "temp": 215.2, "vibration": 0.9, "status": "Operational"},
    {"asset": "TURBINE-T203", "temp": 580.0, "vibration": 4.8, "status": "Maintenance"},
    {"asset": "TURBINE-T203", "temp": 610.0, "vibration": 5.2, "status": "Maintenance"},
    {"asset": "COMP-C300", "temp": 88.0, "vibration": 6.2, "status": "Degraded"},
    {"asset": "COMP-C300", "temp": 92.5, "vibration": 6.8, "status": "Degraded"},
]

# Extract feature limits
temps = [x["temp"] for x in telemetry_data]
vibs = [x["vibration"] for x in telemetry_data]
temp_min, temp_max = min(temps), max(temps)
vib_min, vib_max = min(vibs), max(vibs)

print(f"Normalizing telemetry bounds:")
print(f"  Temperature limits: min={temp_min}, max={temp_max}")
print(f"  Vibration limits: min={vib_min}, max={vib_max}")

In [ ]:
# Feature Normalization (MinMax Scaling)
def scale_features(data, t_min, t_max, v_min, v_max):
    scaled = []
    for item in data:
        n_t = (item["temp"] - t_min) / (t_max - t_min) if t_max > t_min else 0.0
        n_v = (item["vibration"] - v_min) / (v_max - v_min) if v_max > v_min else 0.0
        scaled.append({
            "features": [n_t, n_v],
            "status": item["status"]
        })
    return scaled

scaled_data = scale_features(telemetry_data, temp_min, temp_max, vib_min, vib_max)
print("Example scaled record:")
print(scaled_data[0])

In [ ]:
# Nearest Centroid Model Fitting
class NearestCentroidClassifier:
    def __init__(self):
        self.centroids = {}
        
    def fit(self, dataset):
        grouped = {}
        for row in dataset:
            status = row["status"]
            if status not in grouped:
                grouped[status] = []
            grouped[status].append(row["features"])
            
        for status, vectors in grouped.items():
            arr = np.array(vectors)
            mean_vector = arr.mean(axis=0).tolist()
            self.centroids[status] = mean_vector
            print(f"Calculated centroid for '{status}': {mean_vector}")
            
    def predict(self, features):
        best_status = None
        min_dist = float("inf")
        for status, centroid in self.centroids.items():
            dist = math.sqrt(sum((x - y) ** 2 for x, y in zip(features, centroid)))
            if dist < min_dist:
                min_dist = dist
                best_status = status
        return best_status

model = NearestCentroidClassifier()
model.fit(scaled_data)

In [ ]:
# Evaluate model predictions
correct = 0
for row in scaled_data:
    pred = model.predict(row["features"])
    is_correct = pred == row["status"]
    if is_correct:
        correct += 1
    print(f"Actual: {row['status']:<12} | Predicted: {pred:<12} | Match: {is_correct}")

accuracy = correct / len(scaled_data)
print(f"\nFinal training accuracy: {accuracy * 100:.1f}%")

In [ ]:
# Plot decision boundaries
plt.figure(figsize=(8, 6))

colors = {"Operational": "green", "Maintenance": "orange", "Degraded": "red"}

# Plot training samples
for row in telemetry_data:
    plt.scatter(row["temp"], row["vibration"], color=colors[row["status"]], s=100, edgecolors='black', label=row["status"])

# Plot class centroids (scaled back to original dimensions for visualization)
for status, centroid in model.centroids.items():
    c_temp = centroid[0] * (temp_max - temp_min) + temp_min
    c_vib = centroid[1] * (vib_max - vib_min) + vib_min
    plt.scatter(c_temp, c_vib, color=colors[status], marker='x', s=300, linewidths=3, label=f"{status} Centroid")

# Deduplicate legend labels
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys())

plt.title("Industrial Telemetry Status Centroids & Data Clusters")
plt.xlabel("Temperature (°C)")
plt.ylabel("Vibration (mm/s)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

In [ ]:
# Save model to ml_sandbox/models/failure_predictor.json
model_file = {
    "centroids": model.centroids,
    "temp_bounds": [float(temp_min), float(temp_max)],
    "vib_bounds": [float(vib_min), float(vib_max)]
}

model_dir = "../ml_sandbox/models"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "failure_predictor.json")

with open(model_path, "w") as f:
    json.dump(model_file, f, indent=2)
    
print(f"Model exported successfully to: {os.path.abspath(model_path)}")

## Industrial Knowledge Graph Network Links Mapping

Next, we construct an industrial knowledge graph topology linking:
*   **Equipment Node** (Industrial machines tagged in database)
*   **Document Node** (Indexed technical manuals, safety specifications)
*   **Process Node** (Activities like calibration SOPs and inspections)
*   **Alert Node** (Active anomaly alarms and warnings)

In [ ]:
# Build graph representation
G = nx.Graph()

# Add Node types and labels
nodes = [
    ("PUMP-P102", {"type": "Equipment", "color": "#6366f1"}),
    ("TURBINE-T203", {"type": "Equipment", "color": "#6366f1"}),
    ("BOILER-B401", {"type": "Equipment", "color": "#6366f1"}),
    
    ("pump_ops.pdf", {"type": "Document", "color": "#3b82f6"}),
    ("turbine_safety.pdf", {"type": "Document", "color": "#3b82f6"}),
    ("boiler_spec.xlsx", {"type": "Document", "color": "#3b82f6"}),
    
    ("Valve Calibration SOP", {"type": "Process", "color": "#8b5cf6"}),
    ("Rotor Alignment Check", {"type": "Process", "color": "#8b5cf6"}),
    
    ("Flow Rate Drop Alert", {"type": "Alert", "color": "#ef4444"}),
    ("Overheating Alert", {"type": "Alert", "color": "#ef4444"}),
]

for node, attrs in nodes:
    G.add_node(node, **attrs)

# Add relationship links (edges)
edges = [
    ("PUMP-P102", "pump_ops.pdf", {"label": "DESCRIBES"}),
    ("PUMP-P102", "Valve Calibration SOP", {"label": "MAINTAINED_BY"}),
    ("PUMP-P102", "Flow Rate Drop Alert", {"label": "HAS_ANOMALY"}),
    
    ("TURBINE-T203", "turbine_safety.pdf", {"label": "DESCRIBES"}),
    ("TURBINE-T203", "Rotor Alignment Check", {"label": "MAINTAINED_BY"}),
    ("TURBINE-T203", "Overheating Alert", {"label": "HAS_ANOMALY"}),
    
    ("BOILER-B401", "boiler_spec.xlsx", {"label": "DESCRIBES"}),
]

for u, v, attrs in edges:
    G.add_edge(u, v, **attrs)

print(f"Knowledge Graph structure compiled:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")

In [ ]:
# Visualize the graph using NetworkX & Matplotlib
plt.figure(figsize=(10, 8))

# Spring layout coordinates generation
pos = nx.spring_layout(G, seed=42)

# Extract node colors from node attributes
node_colors = [G.nodes[n]["color"] for n in G.nodes]

# Draw graph elements
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1200, edgecolors="black")
nx.draw_networkx_edges(G, pos, width=2, alpha=0.6, edge_color="gray")
nx.draw_networkx_labels(G, pos, font_size=9, font_family="sans-serif", font_weight="bold")

# Draw connection labels
edge_labels = nx.get_edge_attributes(G, "label")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8, font_color="black")

plt.title("INDUSMIND AI: Asset-Knowledge Graph Network Linkages")
plt.axis("off")
plt.show()